In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
df = pd.read_csv("../dataset/smartgrid_risk_dataset.csv")
print(df.shape)
df.head()

(15000, 27)


,datetime,year,month,day,hour,weekday,temperature,humidity,rainfall,wind_speed,...,upazila,area_type,substation_id,feeder_id,transformer_age,transformer_capacity,outage_history,maintenance_due,population_density,industrial_load_ratio
0,2026-04-13 00:00:00,2026,4,13,0,0,24.62,81.61,0.0,4.78,...,Bahubal,Rural,SS_033,FDR_20,11,200,3,No,1966,0.19
1,2026-04-13 02:00:00,2026,4,13,2,0,23.38,82.84,0.0,4.92,...,Chhatak,Rural,SS_007,FDR_12,7,300,6,No,588,0.22
2,2026-04-13 04:00:00,2026,4,13,4,0,22.85,86.41,0.0,4.82,...,Chunarughat,Rural,SS_015,FDR_04,14,350,1,Yes,1801,0.25
3,2026-04-13 06:00:00,2026,4,13,6,0,23.05,87.09,0.0,5.03,...,Golapganj,Rural,SS_045,FDR_18,13,250,10,No,614,0.07
4,2026-04-13 08:00:00,2026,4,13,8,0,24.15,86.65,0.0,4.79,...,Bishwanath,Rural,SS_016,FDR_18,13,400,2,Yes,1697,0.13


In [3]:
# Drop datetime, risk_score, and date features
df = df.drop(columns=[
    "datetime",
    "risk_score",
    "year",
    "month",
    "day"
])

In [4]:
# Include upazila in categorical columns
categorical_columns = [
    "weather_state",
    "district",
    "upazila",
    "area_type",
    "substation_id",
    "feeder_id",
    "maintenance_due"
]

encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [5]:
target_encoder = LabelEncoder()
df["risk_level"] = target_encoder.fit_transform(df["risk_level"])

In [6]:
X = df.drop("risk_level", axis=1)
y = df["risk_level"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

Logistic Regression Accuracy: 0.8826666666666667
              precision    recall  f1-score   support

           0       0.87      0.83      0.85       798
           1       0.92      0.93      0.92       716
           2       0.87      0.89      0.88      1486

    accuracy                           0.88      3000
   macro avg       0.89      0.88      0.88      3000
weighted avg       0.88      0.88      0.88      3000



In [10]:
dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=8
)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

Decision Tree Accuracy: 0.853
              precision    recall  f1-score   support

           0       0.83      0.78      0.80       798
           1       0.90      0.92      0.91       716
           2       0.85      0.86      0.85      1486

    accuracy                           0.85      3000
   macro avg       0.86      0.85      0.85      3000
weighted avg       0.85      0.85      0.85      3000



In [11]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8653333333333333
              precision    recall  f1-score   support

           0       0.87      0.76      0.81       798
           1       0.92      0.92      0.92       716
           2       0.84      0.90      0.87      1486

    accuracy                           0.87      3000
   macro avg       0.88      0.86      0.86      3000
weighted avg       0.87      0.87      0.86      3000



In [12]:
os.makedirs("../models", exist_ok=True)

joblib.dump(lr, "../models/logistic_regression.pkl")
joblib.dump(dt, "../models/decision_tree.pkl")
joblib.dump(rf, "../models/random_forest.pkl")

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(target_encoder, "../models/target_encoder.pkl")
joblib.dump(encoders, "../models/encoders.pkl")

# Save scaled data for LR/LSTM, unscaled for DT/RF
np.save("../models/X_train.npy", X_train_scaled)
np.save("../models/X_test.npy", X_test_scaled)
np.save("../models/X_test_unscaled.npy", X_test)
np.save("../models/y_train.npy", y_train)
np.save("../models/y_test.npy", y_test)

print("All models saved successfully.")

All models saved successfully.
